In [ ]:
from __future__ import annotations
 
import json
import logging
import time
import sys
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional

ROOT: Path = Path.cwd().parent
sys.path.append(str(ROOT))

from src.core.interface import IYouTubeClient
from api_getter import YouTubeClient

IYouTubeClient.register(YouTubeClient)

client = YouTubeClient(api_key="...")
print(isinstance(client, IYouTubeClient))
# True

In [ ]:
from ast import List

REGION_CODE: str = "BR"
VIDEOS_PER_MACRO: int = 100
MAX_COMMENTS_PER_VIDEO: int = 100
COMMENT_ORDER: str = "relevance"
OUTPUT_DIR: str = "output"
OUTPUT_FORMAT: str = "csv"
SLEEP_BETWEEN_VIDEOS: float = 1

MACROS: List[tuple[str, str]] = [
    ("Educação",                       "educacao"),
    ("Ciência e Tecnologia",           "tecnologia"),
    ("Notícias e Política",            "noticias"),
    ("Entretenimento",                 "entretenimento"),
    ("Música",                         "musica"),
    ("Games",                          "jogos"),
    ("Esportes",                       "esportes"),
    ("Pessoas e Blogs / Howto Style",  "vlog"),
]

_ISO_PATTERN = re.compile(
    r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", re.IGNORECASE
)
 
 
def iso8601_to_seconds(duration: Optional[str]) -> Optional[int]:
    if not duration:
        return None
    m = _ISO_PATTERN.match(duration)
    if not m:
        return None
    h, mn, s = (int(x or 0) for x in m.groups())
    return h * 3600 + mn * 60 + s
 
 
def seconds_to_bucket(seconds: Optional[int]) -> str:
    if seconds is None:
        return "unknown"
    if seconds < 240:
        return "short"
    if seconds <= 1200:
        return "medium"
    return "long"


In [ ]:
@dataclass
class VideoRow:
    video_id: str
    channel_id: str
    channel_title: str
    macro_category: str
    official_category_id: Optional[str]
    official_category_name: Optional[str]
    title: str
    duration_iso: Optional[str]
    duration_seconds: Optional[int]
    duration_bucket: str
    published_at: Optional[str]
    view_count: Optional[int]
    like_count: Optional[int]
    comment_count: Optional[int]
    collected_at: str

 
@dataclass
class CommentRow:
    comment_id: str
    video_id: str
    channel_id: str
    macro_category: str
    text: str
    like_count: int
    published_at: Optional[str]
    collected_at: str
 
 
@dataclass
class RunManifest:
    started_at: str
    finished_at: str
    region: str
    videos_collected: int
    videos_skipped_shorts: int
    comments_collected: int
    by_macro: Dict[str, Dict[str, int]] = field(default_factory=dict)


In [ ]:
class YouTubeEDACollector: 
    def __init__(self, client, output_dir: str = OUTPUT_DIR) -> None:
        self.client = client
        self.out = Path(output_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.logger = self._build_logger()
 
        self._videos: List[VideoRow] = []
        self._comments: List[CommentRow] = []
        self._shorts_skipped: int = 0
  
    @staticmethod
    def _build_logger() -> logging.Logger:
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s [%(levelname)s] %(message)s",
            datefmt="%H:%M:%S",
        )
        return logging.getLogger("YouTubeEDACollector")
 
    @staticmethod
    def _now() -> str:
        return datetime.now(timezone.utc).isoformat()
 
    @staticmethod
    def _safe_int(value) -> Optional[int]:
        try:
            return int(value) if value is not None else None
        except (ValueError, TypeError):
            return None
 
    @staticmethod
    def _is_short(vid) -> bool:
        return str(getattr(vid, "duration", "")).lower() == "short"
 
 
    def _enrich(self, video_id: str) -> tuple[Optional[str], Optional[int], Optional[int], Optional[int]]:
        try:
            detail = self.client.fetch_video_details(
                video_id=video_id,
                parts=["contentDetails", "statistics"],
            )
            if not detail:
                return None, None, None, None
 
            duration_iso = (
                detail.get("contentDetails", {}).get("duration")
            )
            stats = detail.get("statistics", {})
            return (
                duration_iso,
                self._safe_int(stats.get("viewCount")),
                self._safe_int(stats.get("likeCount")),
                self._safe_int(stats.get("commentCount")),
            )
        except Exception as exc:
            self.logger.warning("    Erro detail %s: %s", video_id, exc)
            return None, None, None, None
    
    def _to_row(self, vid, macro: str) -> VideoRow:
        video_id = getattr(vid, "video_id", "") or getattr(vid, "id", "")
 
        duration_iso, view_count, like_count, comment_count = self._enrich(video_id)
        duration_seconds = iso8601_to_seconds(duration_iso)
        duration_bucket = seconds_to_bucket(duration_seconds)
 
        return VideoRow(
            video_id=video_id,
            channel_id=getattr(vid, "channel_id", ""),
            channel_title=getattr(vid, "channel_title", ""),
            macro_category=macro,
            official_category_id=str(getattr(vid, "category_id", "") or ""),
            official_category_name=getattr(vid, "category_name", None),
            title=getattr(vid, "title", ""),
            duration_iso=duration_iso,
            duration_seconds=duration_seconds,
            duration_bucket=duration_bucket,
            published_at=str(getattr(vid, "published_at", "") or ""),
            view_count=view_count,
            like_count=like_count,
            comment_count=comment_count,
            collected_at=self._now(),
        )
 

    def _to_video_row(self, vid, macro: str) -> VideoRow:
        stats = getattr(vid, "statistics", {}) or {}
        return VideoRow(
            video_id=getattr(vid, "video_id", "") or getattr(vid, "id", ""),
            channel_id=getattr(vid, "channel_id", ""),
            macro_category=macro,
            official_category_id=str(getattr(vid, "category_id", "") or ""),
            official_category_name=getattr(vid, "category_name", None),
            title=getattr(vid, "title", ""),
            duration_bucket=str(getattr(vid, "duration", "medium") or "medium"),
            published_at=str(getattr(vid, "published_at", "") or ""),
            view_count=self._safe_int(stats.get("viewCount")),
            like_count=self._safe_int(stats.get("likeCount")),
            comment_count=self._safe_int(stats.get("commentCount")),
            collected_at=self._now(),
        )
 
    def _to_comment_row(self, cmt, video_id: str, channel_id: str, macro: str) -> CommentRow:
        return CommentRow(
            comment_id=getattr(cmt, "comment_id", "") or getattr(cmt, "id", ""),
            video_id=video_id,
            channel_id=channel_id,
            macro_category=macro,
            text=getattr(cmt, "text", "") or getattr(cmt, "text_display", ""),
            like_count=self._safe_int(getattr(cmt, "like_count", 0)) or 0,
            published_at=str(getattr(cmt, "published_at", "") or ""),
            collected_at=self._now(),
        )
  
    # def run(self) -> RunManifest:
    #     started_at = self._now()
    #     self.logger.info(
    #         "iniciando coleta EDA — região=%s | %d macrocategorias",
    #         REGION_CODE, len(MACROS),
    #     )
 
    #     by_macro: Dict[str, Dict[str, int]] = {}
 
    #     for macro_label, category_term in MACROS:
    #         self.logger.info("%s  (termo='%s')", macro_label, category_term)
    #         macro_videos = 0
    #         macro_comments = 0
    #         macro_shorts = 0
 
    #         try:
    #             raw_videos = self.client.fetch_videos_by_category(
    #                 category=category_term,
    #                 region_code=REGION_CODE,
    #                 max_items=VIDEOS_PER_MACRO + 10,
    #                 pages=1,
    #             )
    #         except Exception as exc:
    #             self.logger.warning("  Erro ao buscar vídeos: %s", exc)
    #             raw_videos = []
 
    #         accepted: List[VideoRow] = []
    #         for vid in raw_videos:
    #             if self._is_short(vid):
    #                 macro_shorts += 1
    #                 self._shorts_skipped += 1
    #                 continue
    #             accepted.append(self._to_video_row(vid, macro_label))
    #             if len(accepted) >= VIDEOS_PER_MACRO:
    #                 break
 
    #         self._videos.extend(accepted)
    #         macro_videos = len(accepted)
 
    #         for vr in accepted:
    #             try:
    #                 comments = self.client.fetch_comments(
    #                     video_id=vr.video_id,
    #                     max_pages=2,
    #                     order=COMMENT_ORDER,
    #                     max_items=MAX_COMMENTS_PER_VIDEO,
    #                 )
    #                 rows = [
    #                     self._to_comment_row(c, vr.video_id, vr.channel_id, macro_label)
    #                     for c in comments[:MAX_COMMENTS_PER_VIDEO]
    #                 ]
    #                 self._comments.extend(rows)
    #                 macro_comments += len(rows)
    #                 self.logger.info("    %s → %d comentários", vr.video_id, len(rows))
    #             except Exception as exc:
    #                 self.logger.warning("    Erro comentários %s: %s", vr.video_id, exc)
 
    #             time.sleep(SLEEP_BETWEEN_VIDEOS)
 
    #         by_macro[macro_label] = {
    #             "videos": macro_videos,
    #             "comments": macro_comments,
    #             "shorts_skipped": macro_shorts,
    #         }
    #         self.logger.info(
    #             "vídeos=%d  comentários=%d  shorts_excluídos=%d",
    #             macro_videos, macro_comments, macro_shorts,
    #         )
 
    #     finished_at = self._now()
    #     self._save(by_macro, started_at, finished_at)
 
    #     manifest = RunManifest(
    #         started_at=started_at,
    #         finished_at=finished_at,
    #         region=REGION_CODE,
    #         videos_collected=len(self._videos),
    #         videos_skipped_shorts=self._shorts_skipped,
    #         comments_collected=len(self._comments),
    #         by_macro=by_macro,
    #     )
    #     self.logger.info(
    #         "concluído: %d vídeos | %d comentários | %d Shorts excluídos",
    #         manifest.videos_collected,
    #         manifest.comments_collected,
    #         manifest.videos_skipped_shorts,
    #     )
    #     return manifest

    def run(self) -> Dict:
        started_at = self._now()
        self.logger.info("iniciando coleta — região=%s | %d macros", REGION_CODE, len(MACROS))
 
        by_macro: Dict[str, Dict[str, int]] = {}
 
        for macro_label, category_term in MACROS:
            self.logger.info("%s", macro_label)
            shorts = 0
            accepted: List[VideoRow] = []
 
            try:
                raw = self.client.fetch_videos_by_category(
                    category=category_term,
                    region_code=REGION_CODE,
                    max_items=VIDEOS_PER_MACRO + 10,
                    pages=1,
                )
            except Exception as exc:
                self.logger.warning("erro ao buscar vídeos: %s", exc)
                raw = []
 
            for vid in raw:
                if len(accepted) >= VIDEOS_PER_MACRO:
                    break
 
                # Enriquece para obter duração real antes de decidir se é Short
                row = self._to_row(vid, macro_label)
 
                if row.duration_bucket == "short":
                    shorts += 1
                    self._shorts_skipped += 1
                    self.logger.debug("short excluído: %s (%ds)", row.video_id, row.duration_seconds)
                    continue
 
                accepted.append(row)
                self.logger.info(
                    "    %s | %s | %ds | views=%s",
                    row.video_id, row.duration_bucket,
                    row.duration_seconds or 0, row.view_count,
                )
 
            self._videos.extend(accepted)
 
            unique_channels = len({v.channel_id for v in accepted})
            by_macro[macro_label] = {
                "videos": len(accepted),
                "unique_channels": unique_channels,
                "shorts_skipped": shorts,
            }
            self.logger.info(
                "vídeos=%d  canais_únicos=%d  shorts_excluídos=%d",
                len(accepted), unique_channels, shorts,
            )
 
        finished_at = self._now()
        self._save(by_macro, started_at, finished_at)
        self.logger.info(
            "concluído: %d vídeos | %d canais únicos | %d shorts excluídos",
            len(self._videos),
            len({v.channel_id for v in self._videos}),
            self._shorts_skipped,
        )
        return by_macro
 
 
    def _save(self, by_macro: Dict, started_at: str, finished_at: str) -> None:
        rows = [v.__dict__ for v in self._videos]
 
        if OUTPUT_FORMAT == "json":
            self.client.save_json(rows, str(self.out / "videos_by_channel.json"))
        else:
            self.client.save_csv(rows, str(self.out / "videos_by_channel.csv"))
 
        (self.out / "run_manifest.json").write_text(
            json.dumps(
                {
                    "started_at": started_at,
                    "finished_at": finished_at,
                    "region": REGION_CODE,
                    "total_videos": len(self._videos),
                    "total_unique_channels": len({v.channel_id for v in self._videos}),
                    "total_shorts_skipped": self._shorts_skipped,
                    "by_macro": by_macro,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        self.logger.info("arquivos salvos em %s/", self.out)
 
    # def _save(
    #     self,
    #     by_macro: Dict[str, Dict[str, int]],
    #     started_at: str,
    #     finished_at: str,
    # ) -> None:
    #     videos_dicts   = [v.__dict__ for v in self._videos]
    #     comments_dicts = [c.__dict__ for c in self._comments]
 
    #     if OUTPUT_FORMAT == "json":
    #         self.client.save_json(videos_dicts,   str(self.out / "videos.json"))
    #         self.client.save_json(comments_dicts, str(self.out / "comments.json"))
    #     else:
    #         self.client.save_csv(videos_dicts,   str(self.out / "videos.csv"))
    #         self.client.save_csv(comments_dicts, str(self.out / "comments.csv"))
 
    #     (self.out / "run_manifest.json").write_text(
    #         json.dumps(
    #             {
    #                 "started_at": started_at,
    #                 "finished_at": finished_at,
    #                 "region": REGION_CODE,
    #                 "videos_collected": len(self._videos),
    #                 "videos_skipped_shorts": self._shorts_skipped,
    #                 "comments_collected": len(self._comments),
    #                 "by_macro": by_macro,
    #             },
    #             ensure_ascii=False,
    #             indent=2,
    #         ),
    #         encoding="utf-8",
    #     )
    #     self.logger.info("Arquivos salvos em %s/", self.out)


In [ ]:
client = YouTubeClient()
collector = YouTubeEDACollector(client)
manifest = collector.run()
print("O que foi preciso ser feito, foi feito")